# Paper 1 figures

One figure per cell. Each figure is saved as **PDF (vector, for LaTeX)** and **PNG (300 dpi, for preview)** under `figures_paper1/`, and a suggested English caption is printed and collected in `captions.md`.

Sizes follow the IEEE two-column template: single column 3.5 in, double column 7.16 in. Figures carry no titles; titles belong in the LaTeX caption.

**Requires** the outputs of `dermkg_build.ipynb` (`data/dermkg/`) and `paper1_hierarchical.ipynb` (`results/paper1_hier/`). Cells after Setup are independent, so any single figure can be re-run on its own.

## Setup

In [ ]:
SKINCON_CSV = "image.csv"
F17K_CSV = "fitzpatrick17k.csv"
KG_DIR = "data/dermkg"
RES_DIR = "results/paper1_hier"
FIG_DIR = "figures_paper1"
FORMATS = ("pdf", "png")

In [ ]:
import os, ast, json, itertools, re
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import networkx as nx
from sklearn.metrics import confusion_matrix, recall_score, f1_score, balanced_accuracy_score

os.makedirs(FIG_DIR, exist_ok=True)
SINGLE, DOUBLE = 3.5, 7.16
mpl.rcParams.update({
    "font.family": "serif", "font.serif": ["Times New Roman", "Times", "DejaVu Serif"],
    "font.size": 8, "axes.labelsize": 8, "axes.titlesize": 8, "xtick.labelsize": 7, "ytick.labelsize": 7,
    "legend.fontsize": 7, "legend.frameon": False, "axes.spines.top": False, "axes.spines.right": False,
    "axes.linewidth": 0.6, "xtick.major.width": 0.6, "ytick.major.width": 0.6, "lines.linewidth": 1.2,
    "savefig.bbox": "tight", "savefig.pad_inches": 0.02, "pdf.fonttype": 42, "ps.fonttype": 42,
})
# colorblind-safe palette (Okabe-Ito)
PAL = ["#0072B2", "#E69F00", "#009E73", "#D55E00", "#CC79A7", "#56B4E9", "#F0E442", "#000000", "#999999"]
FAM_COLOR = {"baseline": "#999999", "KG": "#0072B2", "GNN": "#009E73"}
STRENGTH_COLOR = {"strong": "#D55E00", "moderate": "#E69F00", "weak": "#56B4E9", "unsupported": "#999999"}
CAPTIONS = {}


def save(fig, name, caption):
    for ext in FORMATS:
        fig.savefig(os.path.join(FIG_DIR, f"{name}.{ext}"), dpi=300)
    CAPTIONS[name] = caption
    with open(os.path.join(FIG_DIR, "captions.md"), "w") as f:
        for k in sorted(CAPTIONS):
            f.write(f"**{k}**: {CAPTIONS[k]}\n\n")
    plt.show()
    print(f"[{name}] {caption}")

In [ ]:
# data
sk = pd.read_csv(SKINCON_CSV).drop(columns=["Unnamed: 0"], errors="ignore")
sk = sk[sk["Do not consider this image"] == 0].drop(columns=["Do not consider this image"])
sk["md5"] = sk["ImageID"].str.replace(".jpg", "", regex=False)
CONCEPTS = [c for c in sk.columns if c not in ("ImageID", "md5")]
df = sk.merge(pd.read_csv(F17K_CSV), left_on="md5", right_on="md5hash", how="inner")
df = df.merge(pd.read_csv(os.path.join(KG_DIR, "split.csv"))[["md5", "split"]], on="md5").reset_index(drop=True)
df["skin_type"] = df["fitzpatrick_scale"].map(lambda v: f"FST-{int(v)}" if v in range(1, 7) else "FST-unknown")
X = df[CONCEPTS].values.astype(np.float32)
DISEASES = sorted(df["label"].unique()); K = len(DISEASES); D_IDX = {d: i for i, d in enumerate(DISEASES)}
D2CAT = df.groupby("label")["nine_partition_label"].first().to_dict()
CAT2SUP = df.groupby("nine_partition_label")["three_partition_label"].first().to_dict()
CATS = sorted(set(D2CAT.values())); NC = len(CATS); SUPS = sorted(set(CAT2SUP.values()))
d2c = np.array([CATS.index(D2CAT[d]) for d in DISEASES]); c2s = np.array([SUPS.index(CAT2SUP[c]) for c in CATS])
yd = df["label"].map(D_IDX).values; yc = d2c[yd]
tr, va, te = (df["split"].values == s for s in ("train", "val", "test"))
CAT_ORDER = list(pd.Series(yc[tr]).value_counts().index.map(lambda i: CATS[i]))   # by training size
CAT_COLOR = {c: PAL[i % len(PAL)] for i, c in enumerate(CAT_ORDER)}
SHORT = {"inflammatory": "Inflammatory", "malignant epidermal": "Malig. epidermal", "genodermatoses": "Genodermatoses",
         "benign epidermal": "Benign epidermal", "benign dermal": "Benign dermal", "malignant melanoma": "Malig. melanoma",
         "benign melanocyte": "Benign melanocyte", "malignant cutaneous lymphoma": "Cut. lymphoma",
         "malignant dermal": "Malig. dermal"}
CONCEPT_GROUPS = {
    "Primary lesion": ["Macule", "Patch", "Papule", "Plaque", "Nodule", "Vesicle", "Bulla", "Pustule", "Wheal", "Cyst",
                       "Abscess", "Comedo", "Burrow"],
    "Secondary change": ["Scale", "Crust", "Erosion", "Ulcer", "Excoriation", "Fissure", "Atrophy", "Scar",
                         "Lichenification", "Xerosis", "Exudate", "Sclerosis", "Induration", "Friable"],
    "Color": ["Erythema", "Brown(Hyperpigmentation)", "White(Hypopigmentation)", "Purple", "Yellow", "Black", "Blue",
              "Gray", "Salmon", "Pigmented", "Translucent"],
    "Shape/surface": ["Dome-shaped", "Flat topped", "Pedunculated", "Exophytic/Fungating", "Warty/Papillomatous",
                      "Umbilicated", "Acuminate"],
    "Vascular": ["Telangiectasia", "Purpura/Petechiae", "Poikiloderma"],
}
C2G = {c: g for g, cs in CONCEPT_GROUPS.items() for c in cs}
GROUP_COLOR = dict(zip(CONCEPT_GROUPS, PAL))
CSHORT = lambda c: c.replace("(Hyperpigmentation)", "").replace("(Hypopigmentation)", "").replace("Purpura/Petechiae", "Purpura")

# KG artifacts and results
rules = pd.read_csv(os.path.join(KG_DIR, "clinical_rules.csv"))
rules["premise"] = rules["premise"].map(json.loads); rules["targets"] = rules["targets"].map(ast.literal_eval)
triples = pd.read_csv(os.path.join(KG_DIR, "triples_full.csv"))
nodes = pd.read_csv(os.path.join(KG_DIR, "nodes.csv"))
res = pd.read_csv(os.path.join(RES_DIR, "main_results_category.csv")).set_index("model")
ceil = pd.read_csv(os.path.join(RES_DIR, "ceilings.csv"), index_col=0)
summary = json.load(open(os.path.join(RES_DIR, "summary.json")))
probs = np.load(os.path.join(RES_DIR, "test_probs.npz"))
PC = {k[5:]: probs[k] for k in probs.files if k.startswith("cat__")}
PD = {k[5:]: probs[k] for k in probs.files if k.startswith("dis__")}
yct, ydt = probs["y_cat"], probs["y_dis"]
BEST_BASE, BEST_KG = summary["best_baseline"], summary["best_kg"]
MAIN_KG = "KG-Fusion" if "KG-Fusion" in PC else BEST_KG
parse_ci = lambda s: tuple(map(float, re.findall(r"[-0-9.]+", s)))
print(f"{len(df)} images | best baseline: {BEST_BASE} | best KG: {BEST_KG}")

## Fig. 1  Category distribution by split

In [ ]:
cnt = pd.DataFrame({s: pd.Series(yc[m]).value_counts().reindex(range(NC), fill_value=0).values
                    for s, m in (("Train", tr), ("Val", va), ("Test", te))}, index=CATS).loc[CAT_ORDER]
fig, ax = plt.subplots(figsize=(SINGLE, 2.2))
y = np.arange(NC); h = 0.27
for k, (s, col) in enumerate(zip(cnt.columns, ["#0072B2", "#E69F00", "#009E73"])):
    ax.barh(y + (k - 1) * h, cnt[s], h, label=s, color=col)
ax.set_yticks(y); ax.set_yticklabels([SHORT[c] for c in cnt.index]); ax.invert_yaxis()
ax.set_xscale("log"); ax.set_xlabel("Number of images (log scale)"); ax.legend(loc="lower right")
save(fig, "fig01_category_distribution",
     "Distribution of the nine disease categories across the stratified train, validation and test splits (log scale). "
     "Inflammatory conditions account for about 65% of images.")

## Fig. 2  Long-tailed disease distribution

In [ ]:
n = df.groupby("label").size().sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(SINGLE, 2.0))
ax.bar(range(len(n)), n.values, color=[CAT_COLOR[D2CAT[d]] for d in n.index], width=1.0, linewidth=0)
ax.set_xlabel("Disease rank"); ax.set_ylabel("Images"); ax.set_xlim(-1, len(n))
handles = [mpl.patches.Patch(color=CAT_COLOR[c], label=SHORT[c]) for c in CAT_ORDER]
ax.legend(handles=handles, ncol=3, fontsize=5.5, loc="lower center", bbox_to_anchor=(0.5, 1.0), handlelength=1, columnspacing=0.8)
save(fig, "fig02_disease_long_tail",
     f"Long-tailed distribution of the {K} diseases (median {int(n.median())} images per disease), colored by category.")

## Fig. 3  Number of concepts per image

In [ ]:
k = X.sum(1).astype(int)
fig, ax = plt.subplots(figsize=(SINGLE, 1.8))
vals, cnts = np.unique(k, return_counts=True)
ax.bar(vals, cnts, color="#0072B2", width=0.75)
ax.set_xlabel("Annotated concepts per image"); ax.set_ylabel("Images"); ax.set_xticks(vals)
ax.axvline(k.mean(), color="#D55E00", ls="--", lw=0.8); ax.text(k.mean() + 0.15, cnts.max() * 0.9, f"mean {k.mean():.2f}", color="#D55E00")
save(fig, "fig03_concepts_per_image",
     f"Number of dermatologist-annotated concepts per image. Most lesions are described by only two to four of the 48 concepts (mean {k.mean():.2f}).")

## Fig. 4  Concept prevalence by concept group

In [ ]:
prev = pd.Series(X.mean(0), index=CONCEPTS)
order = [c for g in CONCEPT_GROUPS for c in sorted(CONCEPT_GROUPS[g], key=lambda c: -prev[c])]
fig, ax = plt.subplots(figsize=(DOUBLE, 2.0))
ax.bar(range(len(order)), prev[order].values, color=[GROUP_COLOR[C2G[c]] for c in order], width=0.8)
ax.set_xticks(range(len(order))); ax.set_xticklabels([CSHORT(c) for c in order], rotation=70, ha="right", fontsize=6)
ax.set_yscale("log"); ax.set_ylabel("Prevalence (log)"); ax.set_xlim(-0.6, len(order) - 0.4)
ax.legend(handles=[mpl.patches.Patch(color=GROUP_COLOR[g], label=g) for g in CONCEPT_GROUPS], ncol=5, loc="lower center", bbox_to_anchor=(0.5, 1.0))
save(fig, "fig04_concept_prevalence",
     "Prevalence of the 48 SkinCon concepts, grouped by the dermatological lexicon (primary lesion, secondary change, color, "
     "shape/surface, vascular pattern). Prevalence spans more than two orders of magnitude.")

## Fig. 5  Coverage of distinct concept vectors

In [ ]:
key = pd.Series(["".join(map(str, r)) for r in X.astype(int)])
vc = key.value_counts()
cum = np.cumsum(vc.values) / len(key)
fig, ax = plt.subplots(figsize=(SINGLE, 2.0))
ax.plot(np.arange(1, len(vc) + 1), cum, color="#0072B2")
for q in (0.5, 0.8):
    k_ = int(np.searchsorted(cum, q)) + 1
    ax.axhline(q, color="#999999", lw=0.5, ls=":"); ax.plot(k_, q, "o", color="#D55E00", ms=3)
    ax.annotate(f"{k_} vectors cover {q:.0%}", (k_, q), xytext=(k_ * 1.6, q - 0.12), fontsize=6.5,
                arrowprops=dict(arrowstyle="-", lw=0.5, color="#555555"))
ax.set_xscale("log"); ax.set_xlabel(f"Distinct concept vectors, ranked by frequency ({len(vc)} total)")
ax.set_ylabel("Cumulative share of images"); ax.set_ylim(0, 1.02)
save(fig, "fig05_vector_coverage",
     f"Cumulative coverage of the {len(vc)} distinct concept vectors among {len(key)} images. A small number of generic "
     "patterns (e.g. Plaque + Erythema) describe a large fraction of lesions.")

## Fig. 6  Label ambiguity of the most frequent concept vectors

In [ ]:
TOPV = 12
key = np.array(["".join(map(str, r)) for r in X.astype(int)])
vc = pd.Series(key).value_counts().head(TOPV)
comp = pd.crosstab(key, df["nine_partition_label"]).loc[vc.index, CAT_ORDER]
comp = comp.div(comp.sum(1), axis=0)
labels = [" + ".join(CSHORT(CONCEPTS[i]) for i, ch in enumerate(k) if ch == "1") + f"  (n={vc[k]})" for k in vc.index]
fig, ax = plt.subplots(figsize=(DOUBLE, 2.8))
left = np.zeros(TOPV)
for c in CAT_ORDER:
    ax.barh(range(TOPV), comp[c].values, left=left, color=CAT_COLOR[c], label=SHORT[c], height=0.75)
    left += comp[c].values
ax.set_yticks(range(TOPV)); ax.set_yticklabels(labels, fontsize=6.5); ax.invert_yaxis()
ax.set_xlabel("Share of images per category"); ax.set_xlim(0, 1)
ax.legend(ncol=1, loc="center left", bbox_to_anchor=(1.01, 0.5))
save(fig, "fig06_vector_ambiguity",
     f"Category composition of the {TOPV} most frequent concept vectors. Identical concept descriptions map to several "
     "categories, which bounds the accuracy of any concept-only classifier.")

## Fig. 7  Information ceiling of concept-only diagnosis

In [ ]:
lv = ["disease", "category", "super"]
names = {"disease": f"Disease ({K})", "category": f"Category ({NC})", "super": f"Super-category ({len(SUPS)})"}
fig, ax = plt.subplots(figsize=(SINGLE, 2.2))
x = np.arange(3); w = 0.2
bars = [("acc_ceiling", "Accuracy ceiling", "#0072B2"), ("majority_acc", "Majority-class accuracy", "#999999"),
        ("balacc_ceiling", "Balanced-acc. ceiling", "#009E73"), ("chance_balacc", "Chance balanced acc.", "#cccccc")]
for k, (col, lab, c) in enumerate(bars):
    v = ceil.loc[lv, col].values
    ax.bar(x + (k - 1.5) * w, v, w, label=lab, color=c)
    for xi, vi in zip(x + (k - 1.5) * w, v):
        ax.text(xi, vi + 0.01, f"{vi:.2f}", ha="center", fontsize=5.5)
ax.set_xticks(x); ax.set_xticklabels([names[l] for l in lv]); ax.set_ylim(0, 1); ax.set_ylabel("Score")
ax.legend(ncol=2, loc="lower center", bbox_to_anchor=(0.5, 1.0), fontsize=6)
save(fig, "fig07_information_ceiling",
     "Upper bounds of concept-only diagnosis obtained by an in-sample oracle that predicts the best label for each distinct "
     "concept vector, compared with the majority-class and chance baselines at three levels of the hierarchy.")

## Fig. 8  DermKG schema

In [ ]:
nt = nodes["ntype"].value_counts(); et = triples["relation"].value_counts()
POS = {"SuperCategory": (0, 3.0), "DiseaseCategory": (0, 1.5), "Disease": (0, 0), "ClinicalRule": (3.8, 1.5),
       "SkinImage": (3.8, -1.5), "SkinType": (3.8, -3.0), "Concept": (7.6, 0), "ConceptGroup": (7.6, 1.5)}
COL = {"SkinImage": "#999999", "Concept": "#0072B2", "ConceptGroup": "#56B4E9", "Disease": "#D55E00",
       "DiseaseCategory": "#E69F00", "SuperCategory": "#CC79A7", "SkinType": "#F0E442", "ClinicalRule": "#009E73"}
BW, BH = 2.2, 0.66
STRUCT = {"HAS_CONCEPT", "HAS_SKIN_TYPE", "DIAGNOSED_AS", "BELONGS_TO", "IS_A", "SUBCLASS_OF"}
# (head, tail, relation, label position along the edge)
E = [("Disease", "DiseaseCategory", "IS_A", 0.5), ("DiseaseCategory", "SuperCategory", "SUBCLASS_OF", 0.5),
     ("SkinImage", "Disease", "DIAGNOSED_AS", 0.5), ("SkinImage", "Concept", "HAS_CONCEPT", 0.5),
     ("SkinImage", "SkinType", "HAS_SKIN_TYPE", 0.5), ("Concept", "ConceptGroup", "BELONGS_TO", 0.5),
     ("Disease", "Concept", "CHARACTERIZED_BY", 0.27), ("Concept", "ClinicalRule", "GOVERNED_BY", 0.33),
     ("ClinicalRule", "Disease", "INDICATES", 0.5), ("SkinImage", "ClinicalRule", "ACTIVATES", 0.8)]

def border(p, q):
    """point on the box around p in the direction of q"""
    (x1, y1), (x2, y2) = p, q; dx, dy = x2 - x1, y2 - y1
    t = min(BW / 2 / abs(dx) if dx else np.inf, BH / 2 / abs(dy) if dy else np.inf)
    return x1 + dx * t, y1 + dy * t

fig, ax = plt.subplots(figsize=(DOUBLE, 3.4))
for h, t, r, lp in E:
    p, q = border(POS[h], POS[t]), border(POS[t], POS[h])
    c = "#777777" if r in STRUCT else "#0072B2"
    ax.add_patch(mpl.patches.FancyArrowPatch(p, q, arrowstyle="-|>", mutation_scale=9, lw=0.9, color=c,
                                             linestyle="-" if r in STRUCT else (0, (4, 2))))
    lx, ly = p[0] + (q[0] - p[0]) * lp, p[1] + (q[1] - p[1]) * lp
    if abs(q[0] - p[0]) < 1e-6 and r != "ACTIVATES":     # short vertical edge: label beside the arrow
        ax.text(lx + 0.1, ly, f"{r} ({et.get(r, 0):,})", fontsize=5.8, ha="left", va="center", color=c)
    else:
        ax.text(lx, ly, f"{r}\n{et.get(r, 0):,}", fontsize=5.8, ha="center", va="center", color=c,
                bbox=dict(boxstyle="round,pad=0.12", fc="white", ec="none"))
cx, cy = POS["Concept"]
ax.add_patch(mpl.patches.FancyArrowPatch((cx + BW / 2, cy + 0.15), (cx + BW / 2, cy - 0.15), connectionstyle="arc3,rad=-2.2",
                                         arrowstyle="-|>", mutation_scale=8, lw=0.9, color="#0072B2", linestyle=(0, (4, 2))))
ax.text(cx + BW / 2 + 0.55, cy, f"CO_OCCURS\n{et.get('CO_OCCURS', 0):,}", fontsize=5.8, ha="left", va="center", color="#0072B2")
for n, (x_, y_) in POS.items():
    ax.add_patch(mpl.patches.FancyBboxPatch((x_ - BW / 2, y_ - BH / 2), BW, BH, boxstyle="round,pad=0.02,rounding_size=0.12",
                                            fc=COL[n], ec="white", lw=1))
    dark = n not in ("SkinType", "ConceptGroup")
    ax.text(x_, y_ + 0.1, n, ha="center", va="center", fontsize=6.6 if len(n) < 14 else 6.0, fontweight="bold", color="white" if dark else "black")
    ax.text(x_, y_ - 0.16, f"{nt.get(n, 0):,} nodes", ha="center", va="center", fontsize=6, color="white" if dark else "black")
ax.legend(handles=[mpl.lines.Line2D([], [], color="#777777", lw=0.9, label="Structural relation"),
                   mpl.lines.Line2D([], [], color="#0072B2", lw=0.9, ls=(0, (4, 2)), label="Knowledge / inference relation")],
          loc="upper right", bbox_to_anchor=(1.0, 0.98))
ax.set_xlim(-1.2, 10.0); ax.set_ylim(-3.5, 3.5); ax.set_aspect("equal"); ax.axis("off")
save(fig, "fig08_kg_schema",
     f"DermKG schema with {len(nt)} node types and {len(et)} relation types (edge counts on arrows). Solid gray: structural relations read from annotations; "
     "dashed blue: knowledge and inference relations derived from training statistics and clinical rules.")

## Fig. 9  Nodes by type

In [ ]:
nt = nodes["ntype"].value_counts().sort_values()
fig, ax = plt.subplots(figsize=(SINGLE, 1.9))
ax.barh(nt.index, nt.values, color="#0072B2")
for i, v in enumerate(nt.values):
    ax.text(v * 1.1, i, f"{v:,}", va="center", fontsize=6.5)
ax.set_xscale("log"); ax.set_xlabel("Nodes (log scale)"); ax.set_xlim(1, nt.max() * 5)
save(fig, "fig09_nodes_by_type", f"Number of DermKG nodes per type ({len(nodes):,} nodes in total).")

## Fig. 10  Edges by relation

In [ ]:
et = triples["relation"].value_counts().sort_values()
struct = {"HAS_CONCEPT", "HAS_SKIN_TYPE", "DIAGNOSED_AS", "BELONGS_TO", "IS_A", "SUBCLASS_OF"}
fig, ax = plt.subplots(figsize=(SINGLE, 2.2))
ax.barh(et.index, et.values, color=["#999999" if r in struct else "#0072B2" for r in et.index])
for i, v in enumerate(et.values):
    ax.text(v * 1.1, i, f"{v:,}", va="center", fontsize=6.5)
ax.set_xscale("log"); ax.set_xlabel("Edges (log scale)"); ax.set_xlim(1, et.max() * 5)
ax.legend(handles=[mpl.patches.Patch(color="#999999", label="Structural"), mpl.patches.Patch(color="#0072B2", label="Knowledge / inference")],
          loc="lower right")
save(fig, "fig10_edges_by_relation", f"Number of DermKG edges per relation type ({len(triples):,} edges in total).")

## Fig. 11  Degree distribution

In [ ]:
deg = pd.concat([triples["head"], triples["tail"]]).value_counts()
ntype = nodes.set_index("id")["ntype"]
fig, ax = plt.subplots(figsize=(SINGLE, 2.1))
for t, c in (("SkinImage", "#999999"), ("Concept", "#0072B2"), ("Disease", "#D55E00"), ("ClinicalRule", "#009E73")):
    d = deg[deg.index.map(ntype) == t]
    ax.scatter(np.arange(1, len(d) + 1), d.sort_values(ascending=False).values, s=3, color=c, label=t)
ax.set_xscale("log"); ax.set_yscale("log"); ax.set_xlabel("Node rank within type"); ax.set_ylabel("Degree")
ax.legend(markerscale=2)
save(fig, "fig11_degree_distribution", "Rank-degree plot of DermKG nodes by type (log-log). Concept nodes act as high-degree hubs linking images, diseases and rules.")

## Fig. 12  Disease-concept profile matrix (full)

In [ ]:
P = pd.read_csv(os.path.join(KG_DIR, "disease_concept_prob_train.csv"), index_col=0)
order_d = sorted(DISEASES, key=lambda d: (CAT_ORDER.index(D2CAT[d]), d))
order_c = [c for g in CONCEPT_GROUPS for c in sorted(CONCEPT_GROUPS[g], key=lambda c: -X[:, CONCEPTS.index(c)].mean())]
fig, ax = plt.subplots(figsize=(DOUBLE, 8.6))
im = ax.imshow(P.loc[order_d, order_c].values, aspect="auto", cmap="viridis", vmin=0, vmax=1)
ax.set_xticks(range(len(order_c))); ax.set_xticklabels([CSHORT(c) for c in order_c], rotation=90, fontsize=5.5)
ax.set_yticks(range(len(order_d))); ax.set_yticklabels(order_d, fontsize=4.2)
for xb in np.cumsum([len(CONCEPT_GROUPS[g]) for g in CONCEPT_GROUPS])[:-1]:
    ax.axvline(xb - 0.5, color="white", lw=0.8)
for yb in [i for i in range(1, len(order_d)) if D2CAT[order_d[i]] != D2CAT[order_d[i - 1]]]:
    ax.axhline(yb - 0.5, color="white", lw=0.8)
ax.spines[:].set_visible(False)
cb = plt.colorbar(im, ax=ax, fraction=0.025, pad=0.01); cb.set_label("Smoothed P(concept | disease)")
save(fig, "fig12_disease_concept_matrix",
     "DermKG disease-to-concept profiles estimated on the training split. Columns are grouped by concept group and rows by disease category (white lines).")

## Fig. 13  Category-concept profile matrix (compact)

In [ ]:
Pc = pd.DataFrame(X[tr], columns=CONCEPTS).groupby(df.loc[tr, "nine_partition_label"].values).mean()
p_glob = X[tr].mean(0)
lift = np.log2((Pc + 1e-3) / (p_glob + 1e-3))
keep = [c for c in CONCEPTS if X[:, CONCEPTS.index(c)].sum() >= 30]
keep = [c for g in CONCEPT_GROUPS for c in CONCEPT_GROUPS[g] if c in keep]
fig, ax = plt.subplots(figsize=(DOUBLE, 2.6))
v = lift.loc[CAT_ORDER, keep].values
im = ax.imshow(v, aspect="auto", cmap="RdBu_r", vmin=-3, vmax=3)
ax.set_xticks(range(len(keep))); ax.set_xticklabels([CSHORT(c) for c in keep], rotation=70, ha="right", fontsize=6)
ax.set_yticks(range(NC)); ax.set_yticklabels([SHORT[c] for c in CAT_ORDER])
for xb in np.cumsum([sum(c in keep for c in CONCEPT_GROUPS[g]) for g in CONCEPT_GROUPS])[:-1]:
    ax.axvline(xb - 0.5, color="black", lw=0.5)
cb = plt.colorbar(im, ax=ax, fraction=0.02, pad=0.01); cb.set_label("log2 lift over global prevalence")
save(fig, "fig13_category_concept_lift",
     "Concept enrichment per category (log2 lift of category-level prevalence over global prevalence, training split; concepts with at least 30 images). "
     "Red marks concepts characteristic of a category.")

## Fig. 14  Concept co-occurrence (PMI)

In [ ]:
Xt = X[tr]; N = len(Xt)
keep = [c for g in CONCEPT_GROUPS for c in CONCEPT_GROUPS[g] if X[:, CONCEPTS.index(c)].sum() >= 30]
idx = [CONCEPTS.index(c) for c in keep]
co = Xt[:, idx].T @ Xt[:, idx]; p = co.diagonal() / N
with np.errstate(divide="ignore"):
    pmi = np.log((co / N) / np.outer(p, p))
pmi[co < 5] = np.nan; np.fill_diagonal(pmi, np.nan)
fig, ax = plt.subplots(figsize=(SINGLE, 3.2))
im = ax.imshow(pmi, cmap="RdBu_r", vmin=-2, vmax=2)
ax.set_xticks(range(len(keep))); ax.set_xticklabels([CSHORT(c) for c in keep], rotation=90, fontsize=5)
ax.set_yticks(range(len(keep))); ax.set_yticklabels([CSHORT(c) for c in keep], fontsize=5)
cb = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.02); cb.set_label("PMI")
save(fig, "fig14_concept_pmi",
     "Pointwise mutual information between concepts on the training split (pairs with fewer than 5 co-occurrences masked). "
     "Positive values underlie the CO_OCCURS relation.")

## Fig. 15  Expert rule validation (lift vs support)

In [ ]:
er = rules[rules.origin == "expert"]
fig, ax = plt.subplots(figsize=(SINGLE, 2.4))
for s, g in er.groupby("strength"):
    ax.scatter(g["support"], g["lift"], s=22, color=STRENGTH_COLOR[s], label=s.capitalize(), edgecolors="white", lw=0.4, zorder=3)
for _, r in er.iterrows():
    ax.annotate(r["rule_id"], (r["support"], r["lift"]), xytext=(3, 2), textcoords="offset points", fontsize=5.5)
ax.axhline(5, color="#999999", ls="--", lw=0.6); ax.axhline(2, color="#999999", ls=":", lw=0.6)
ax.axvline(10, color="#999999", ls="--", lw=0.6)
ax.set_xscale("log"); ax.set_yscale("log"); ax.set_xlabel("Support (training images satisfying the premise)")
ax.set_ylabel("Lift"); ax.legend(loc="upper right")
save(fig, "fig15_rule_validation",
     f"Training-split validation of the {len(er)} expert clinical rules. Dashed lines mark the strong-rule thresholds (lift 5, support 10) "
     "and the dotted line the moderate threshold (lift 2). All rules have lift above 1.")

## Fig. 16  Expert rule precision and recall

In [ ]:
er = rules[rules.origin == "expert"].sort_values("lift", ascending=False)
fig, ax = plt.subplots(figsize=(DOUBLE, 2.0))
x = np.arange(len(er)); w = 0.38
ax.bar(x - w / 2, er["precision"], w, label="Precision", color="#0072B2")
ax.bar(x + w / 2, er["recall"], w, label="Recall", color="#E69F00")
ax.set_xticks(x); ax.set_xticklabels(er["rule_id"]); ax.set_ylim(0, 1.08); ax.set_ylabel("Score (training split)")
for xi, s in zip(x, er["strength"]):
    ax.text(xi, 1.02, s[0].upper(), ha="center", fontsize=6, color=STRENGTH_COLOR[s])
ax.legend(ncol=2, loc="upper right", bbox_to_anchor=(1, 0.95))
save(fig, "fig16_rule_precision_recall",
     "Precision and recall of each expert rule with respect to its target diseases on the training split, sorted by lift. "
     "Letters give the assigned strength (S strong, M moderate).")

## Fig. 17  Rule coverage by category

In [ ]:
def premise_mask(Xq, premise):
    m = np.ones(len(Xq), dtype=bool)
    for clause in premise:
        m &= Xq[:, [CONCEPTS.index(c) for c in clause]].sum(1) > 0
    return m
ACT = np.stack([premise_mask(X, p) for p in rules["premise"]], 1)
exp_ = (rules.origin == "expert").values
cov = pd.DataFrame({"Expert rules": ACT[:, exp_].any(1), "Expert + mined": ACT.any(1),
                    "cat": df["nine_partition_label"].values}).groupby("cat").mean().loc[CAT_ORDER]
fig, ax = plt.subplots(figsize=(SINGLE, 2.2))
y = np.arange(NC); h = 0.38
ax.barh(y - h / 2, cov["Expert rules"], h, color="#0072B2", label="Expert rules")
ax.barh(y + h / 2, cov["Expert + mined"], h, color="#56B4E9", label="Expert + mined")
ax.set_yticks(y); ax.set_yticklabels([SHORT[c] for c in CAT_ORDER]); ax.invert_yaxis()
ax.set_xlim(0, 1); ax.set_xlabel("Images activating at least one rule"); ax.legend(ncol=2, loc="lower center", bbox_to_anchor=(0.5, 1.0))
save(fig, "fig17_rule_coverage", "Share of images that activate at least one clinical rule, by category.")

## Fig. 18  Main results: macro-F1 with 95% CI

In [ ]:
r = res.sort_values("macroF1")
ci = np.array([parse_ci(s) for s in r["F1_CI95"]])
fig, ax = plt.subplots(figsize=(SINGLE, 3.4))
y = np.arange(len(r))
ax.barh(y, r["macroF1"], color=[FAM_COLOR[f] for f in r["family"]], height=0.7)
ax.errorbar(r["macroF1"], y, xerr=[r["macroF1"] - ci[:, 0], ci[:, 1] - r["macroF1"]], fmt="none", ecolor="black", elinewidth=0.6, capsize=1.5)
ax.set_yticks(y); ax.set_yticklabels(r.index, fontsize=6); ax.set_xlabel("Macro-F1 (test, 9 categories)")
ax.legend(handles=[mpl.patches.Patch(color=c, label=f) for f, c in FAM_COLOR.items() if f in set(r["family"])], loc="lower right")
save(fig, "fig18_main_macro_f1",
     "Category-level macro-F1 on the test set with 95% bootstrap confidence intervals (1,000 resamples). Gray: data-only baselines; blue: KG-based models.")

## Fig. 19  Main results: balanced accuracy with 95% CI

In [ ]:
r = res.sort_values("balAcc")
ci = np.array([parse_ci(s) for s in r["balAcc_CI95"]])
fig, ax = plt.subplots(figsize=(SINGLE, 3.4))
y = np.arange(len(r))
ax.barh(y, r["balAcc"], color=[FAM_COLOR[f] for f in r["family"]], height=0.7)
ax.errorbar(r["balAcc"], y, xerr=[r["balAcc"] - ci[:, 0], ci[:, 1] - r["balAcc"]], fmt="none", ecolor="black", elinewidth=0.6, capsize=1.5)
ax.axvline(ceil.loc["category", "chance_balacc"], color="#999999", ls=":", lw=0.8)
ax.set_yticks(y); ax.set_yticklabels(r.index, fontsize=6); ax.set_xlabel("Balanced accuracy (test, 9 categories)")
save(fig, "fig19_main_balanced_accuracy",
     "Category-level balanced accuracy on the test set with 95% bootstrap confidence intervals. The dotted line marks chance (1/9).")

## Fig. 20  Results relative to the ceiling

In [ ]:
sel = [BEST_BASE, "KG-Mixture", MAIN_KG]
sel = list(dict.fromkeys(s for s in sel if s in res.index))
fig, ax = plt.subplots(figsize=(SINGLE, 2.0))
metrics = [("acc", "Accuracy", "acc_ceiling", "majority_acc"), ("balAcc", "Balanced accuracy", "balacc_ceiling", "chance_balacc")]
x = np.arange(2); w = 0.8 / len(sel)
for k, n in enumerate(sel):
    ax.bar(x + (k - (len(sel) - 1) / 2) * w, [res.loc[n, m[0]] for m in metrics], w, label=n,
           color=["#999999", "#56B4E9", "#0072B2"][k % 3])
for i, m in enumerate(metrics):
    ax.hlines(ceil.loc["category", m[2]], i - 0.45, i + 0.45, color="#D55E00", lw=1.2, label="Ceiling" if i == 0 else None)
    ax.hlines(ceil.loc["category", m[3]], i - 0.45, i + 0.45, color="#000000", lw=0.8, ls=":", label="Majority / chance" if i == 0 else None)
ax.set_xticks(x); ax.set_xticklabels([m[1] for m in metrics]); ax.set_ylim(0, 1)
ax.legend(fontsize=6, loc="center left", bbox_to_anchor=(1.0, 0.5))
save(fig, "fig20_results_vs_ceiling",
     "Test performance of the best baseline and the KG models relative to the concept-only ceiling (red) and the majority-class / chance reference (dotted).")

## Fig. 21  Fusion sensitivity

In [ ]:
gp = os.path.join(RES_DIR, "fusion_grid.csv")
assert os.path.exists(gp), ("fusion_grid.csv missing: in paper1_hierarchical.ipynb run "
                            "GRID_RES.to_csv(os.path.join(OUT_DIR, 'fusion_grid.csv'), index=False)")
g = pd.read_csv(gp)
fig, ax = plt.subplots(figsize=(SINGLE, 2.0))
for k, (lam, gg) in enumerate(g.groupby("lambda")):
    gg = gg.sort_values("alpha")
    ax.plot(gg["alpha"], gg["valF1"], marker="o", ms=2.5, color=PAL[k], label=f"$\\lambda$={lam}")
ax.axvline(summary["fusion"]["alpha"], color="#999999", ls=":", lw=0.8)
ax.set_xlabel(r"$\alpha$ (weight of KG-Mixture)"); ax.set_ylabel("Validation macro-F1"); ax.legend(ncol=2)
save(fig, "fig21_fusion_sensitivity",
     r"Sensitivity of KG-Fusion to the mixture weight $\alpha$ and the rule weight $\lambda$ on the validation split. The dotted line marks the selected $\alpha$.")

## Fig. 22  Per-category recall

In [ ]:
sel = list(dict.fromkeys(s for s in [BEST_BASE, "KG-Mixture", MAIN_KG] if s in PC))
rec = pd.DataFrame({n: recall_score(yct, PC[n].argmax(1), labels=range(NC), average=None, zero_division=0) for n in sel}, index=CATS).loc[CAT_ORDER]
fig, ax = plt.subplots(figsize=(DOUBLE, 2.1))
x = np.arange(NC); w = 0.8 / len(sel)
for k, n in enumerate(sel):
    ax.bar(x + (k - (len(sel) - 1) / 2) * w, rec[n], w, label=n, color=["#999999", "#56B4E9", "#0072B2"][k % 3])
ntr = pd.Series(yc[tr]).value_counts()
ax.set_xticks(x); ax.set_xticklabels([f"{SHORT[c]}\n(n={ntr[CATS.index(c)]})" for c in CAT_ORDER], fontsize=6)
ax.set_ylabel("Recall (test)"); ax.set_ylim(0, 1); ax.legend(ncol=3, loc="lower center", bbox_to_anchor=(0.5, 1.0))
save(fig, "fig22_per_category_recall", "Per-category test recall, categories sorted by training size (n = training images).")

## Fig. 23  Category confusion: best baseline

In [ ]:
n = BEST_BASE
cm = confusion_matrix(yct, PC[n].argmax(1), labels=[CATS.index(c) for c in CAT_ORDER], normalize="true")
fig, ax = plt.subplots(figsize=(SINGLE, 3.0))
im = ax.imshow(cm, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(NC)); ax.set_xticklabels([SHORT[c] for c in CAT_ORDER], rotation=60, ha="right", fontsize=6)
ax.set_yticks(range(NC)); ax.set_yticklabels([SHORT[c] for c in CAT_ORDER], fontsize=6)
for i, j in itertools.product(range(NC), repeat=2):
    if cm[i, j] >= 0.005:
        ax.text(j, i, f"{cm[i, j]:.2f}", ha="center", va="center", fontsize=5, color="white" if cm[i, j] > 0.5 else "black")
ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.spines[:].set_visible(False)
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.02)
save(fig, "fig23_confusion_baseline",
     f"Row-normalized category confusion matrix of {n} on the test set.")

## Fig. 24  Category confusion: KG-Fusion

In [ ]:
n = MAIN_KG
cm = confusion_matrix(yct, PC[n].argmax(1), labels=[CATS.index(c) for c in CAT_ORDER], normalize="true")
fig, ax = plt.subplots(figsize=(SINGLE, 3.0))
im = ax.imshow(cm, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(NC)); ax.set_xticklabels([SHORT[c] for c in CAT_ORDER], rotation=60, ha="right", fontsize=6)
ax.set_yticks(range(NC)); ax.set_yticklabels([SHORT[c] for c in CAT_ORDER], fontsize=6)
for i, j in itertools.product(range(NC), repeat=2):
    if cm[i, j] >= 0.005:
        ax.text(j, i, f"{cm[i, j]:.2f}", ha="center", va="center", fontsize=5, color="white" if cm[i, j] > 0.5 else "black")
ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.spines[:].set_visible(False)
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.02)
save(fig, "fig24_confusion_kgfusion",
     f"Row-normalized category confusion matrix of {n} on the test set.")

## Fig. 25  Where the errors go

In [ ]:
sel = list(dict.fromkeys(s for s in [BEST_BASE, MAIN_KG] if s in PC))
fig, ax = plt.subplots(figsize=(SINGLE, 2.2))
infl = CATS.index("inflammatory"); rows = [c for c in CAT_ORDER if c != "inflammatory"]
y = np.arange(len(rows)); h = 0.38
for k, n in enumerate(sel):
    pred = PC[n].argmax(1)
    share = [((yct == CATS.index(c)) & (pred == infl)).sum() / max((yct == CATS.index(c)).sum(), 1) for c in rows]
    ax.barh(y + (k - 0.5) * h, share, h, label=n, color=["#999999", "#0072B2"][k])
ax.set_yticks(y); ax.set_yticklabels([SHORT[c] for c in rows]); ax.invert_yaxis()
ax.set_xlim(0, 1); ax.set_xlabel("Share of test images misclassified as Inflammatory"); ax.legend(ncol=2, loc="lower center", bbox_to_anchor=(0.5, 1.0))
save(fig, "fig25_errors_to_inflammatory",
     "Fraction of each non-inflammatory category predicted as Inflammatory. Collapse toward the majority category is the dominant error mode.")

## Fig. 26  Super-category confusion

In [ ]:
n = MAIN_KG
pc = PC[n].argmax(1)
cm = confusion_matrix(c2s[yct], c2s[pc], labels=range(len(SUPS)), normalize="true")
fig, ax = plt.subplots(figsize=(SINGLE * 0.75, 2.0))
im = ax.imshow(cm, cmap="Blues", vmin=0, vmax=1)
lab = [s.capitalize() for s in SUPS]
ax.set_xticks(range(len(SUPS))); ax.set_xticklabels(lab, rotation=30, ha="right"); ax.set_yticks(range(len(SUPS))); ax.set_yticklabels(lab)
for i, j in itertools.product(range(len(SUPS)), repeat=2):
    ax.text(j, i, f"{cm[i, j]:.2f}", ha="center", va="center", fontsize=6.5, color="white" if cm[i, j] > 0.5 else "black")
ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.spines[:].set_visible(False)
save(fig, "fig26_super_confusion", f"Super-category confusion of {n} on the test set, obtained by aggregating category predictions along SUBCLASS_OF.")

## Fig. 27  Fine-grained differential diagnosis: top-k accuracy

In [ ]:
fig, ax = plt.subplots(figsize=(SINGLE, 2.0))
ks = np.arange(1, 11)
for k_, (n, P) in enumerate(PD.items()):
    order = np.argsort(-P, 1)
    acc = [(order[:, :k] == ydt[:, None]).any(1).mean() for k in ks]
    ax.plot(ks, acc, marker="o", ms=2.5, label=n, color=PAL[k_ % len(PAL)], lw=1.6 if n.startswith("KG") else 0.9)
ax.axhline(ceil.loc["disease", "acc_ceiling"], color="#D55E00", ls="--", lw=0.7)
ax.text(1, ceil.loc["disease", "acc_ceiling"] + 0.012, "top-1 ceiling", ha="left", fontsize=6, color="#D55E00")
ax.set_xlabel("k"); ax.set_ylabel(f"Top-k accuracy ({K} diseases)"); ax.set_xticks(ks); ax.legend(fontsize=5.5, loc="center left", bbox_to_anchor=(1.0, 0.5))
save(fig, "fig27_topk_disease", f"Top-k accuracy of fine-grained diagnosis over {K} diseases (differential diagnosis setting). The dashed line is the top-1 concept-only ceiling.")

## Fig. 28  Skin-type breakdown

In [ ]:
sel = list(dict.fromkeys(s for s in [BEST_BASE, MAIN_KG] if s in PC))
st = df.loc[te, "skin_type"].values
types = [t for t in sorted(set(st)) if t != "FST-unknown"]
fig, ax = plt.subplots(figsize=(SINGLE, 2.0))
x = np.arange(len(types)); w = 0.38
for k, n in enumerate(sel):
    pred = PC[n].argmax(1)
    v = [balanced_accuracy_score(yct[st == t], pred[st == t]) for t in types]
    ax.bar(x + (k - 0.5) * w, v, w, label=n, color=["#999999", "#0072B2"][k])
cnt = Counter(st)
ax.set_xticks(x); ax.set_xticklabels([f"{t.split('-')[1]}\n(n={cnt[t]})" for t in types])
ax.set_xlabel("Fitzpatrick skin type"); ax.set_ylabel("Balanced accuracy (test)"); ax.legend(loc="upper left")
save(fig, "fig28_skin_type", "Category-level balanced accuracy by Fitzpatrick skin type (test set; n = test images).")

## Fig. 29  Rule-backed vs. unsupported predictions

In [ ]:
def premise_mask(Xq, premise):
    m = np.ones(len(Xq), dtype=bool)
    for clause in premise:
        m &= Xq[:, [CONCEPTS.index(c) for c in clause]].sum(1) > 0
    return m
Xte = X[te]; exp_ = (rules.origin == "expert").values
A = np.stack([premise_mask(Xte, p) for p in rules["premise"]], 1)
tcat = np.zeros((len(rules), NC), bool)
for k, tg in enumerate(rules["targets"]):
    tcat[k, [d2c[D_IDX[d]] for d in tg]] = True
pred = PC[MAIN_KG].argmax(1)
backed = np.array([(A[i] & exp_ & tcat[:, pred[i]]).any() for i in range(len(pred))])
vals = {"Rule-backed": (backed, "#009E73"), "Not rule-backed": (~backed, "#999999")}
fig, ax = plt.subplots(figsize=(SINGLE * 0.8, 1.9))
for k, (lab, (m, c)) in enumerate(vals.items()):
    v = balanced_accuracy_score(yct[m], pred[m]); a = (pred[m] == yct[m]).mean()
    ax.bar([k - 0.18], [a], 0.34, color=c, alpha=0.6); ax.bar([k + 0.18], [v], 0.34, color=c)
    ax.text(k, max(a, v) + 0.03, f"n={m.sum()}", ha="center", fontsize=6)
ax.set_xticks([0, 1]); ax.set_xticklabels(list(vals)); ax.set_ylim(0, 1); ax.set_ylabel("Test score")
ax.legend(handles=[mpl.patches.Patch(color="#555555", alpha=0.6, label="Accuracy"), mpl.patches.Patch(color="#555555", label="Balanced acc.")], loc="upper right")
save(fig, "fig29_rule_backed", f"Test performance of {MAIN_KG} for predictions supported by at least one activated expert rule versus unsupported predictions.")

## Fig. 30  Case-level explanation

In [ ]:
# recompute KG-Mixture disease profiles with the selected hyperparameters
a2 = summary["kg_mixture"]["alpha2"]; a1 = a0 = 5.0
Xtr, ytr_d = X[tr], yd[tr]; ytr_c = d2c[ytr_d]; p_glob = Xtr.mean(0)
cnt_d, cnt_c = np.bincount(ytr_d, minlength=K), np.bincount(ytr_c, minlength=NC)
cnt_s = np.bincount(c2s[ytr_c], minlength=len(SUPS))
p_s = (np.eye(len(SUPS))[c2s[ytr_c]].T @ Xtr + a0 * p_glob) / (cnt_s[:, None] + a0)
p_k = (np.eye(NC)[ytr_c].T @ Xtr + a1 * p_s[c2s]) / (cnt_c[:, None] + a1)
P_D = np.clip((np.eye(K)[ytr_d].T @ Xtr + a2 * p_k[d2c]) / (cnt_d[:, None] + a2), 1e-4, 1 - 1e-4)
Xte = X[te]; P = PC[MAIN_KG]; pred = P.argmax(1)
cands = np.flatnonzero((pred == yct) & (yct != CATS.index("inflammatory")) & (Xte.sum(1) >= 3))
i = cands[0]
x = Xte[i]; k1, k2 = np.argsort(-P[i])[:2]
ll = x @ np.log(P_D).T + (1 - x) @ np.log(1 - P_D).T + np.log(np.maximum(cnt_d, 1))
d1 = np.flatnonzero(d2c == k1)[ll[d2c == k1].argmax()]; d2 = np.flatnonzero(d2c == k2)[ll[d2c == k2].argmax()]
delta = x * np.log(P_D[d1] / P_D[d2]) + (1 - x) * np.log((1 - P_D[d1]) / (1 - P_D[d2]))
order = np.argsort(-np.abs(delta))[:7][::-1]
fig, ax = plt.subplots(figsize=(SINGLE, 2.0))
ax.barh([f"{CSHORT(CONCEPTS[j])} ({'present' if x[j] else 'absent'})" for j in order], delta[order],
        color=["#009E73" if v > 0 else "#D55E00" for v in delta[order]])
ax.axvline(0, color="black", lw=0.5); ax.set_xlabel(f"Log-evidence: {DISEASES[d1]} vs. {DISEASES[d2]}")
save(fig, "fig30_case_explanation",
     f"Concept-level explanation for a test lesion (true: {DISEASES[ydt[i]]}). KG-Mixture attributes the predicted category "
     f"({SHORT[CATS[k1]]}, via {DISEASES[d1]}) against the runner-up ({SHORT[CATS[k2]]}, via {DISEASES[d2]}).")

## Fig. 31  Calibration (reliability diagram)

In [ ]:
sel = list(dict.fromkeys(s for s in [BEST_BASE, "KG-Mixture", MAIN_KG] if s in PC))
fig, ax = plt.subplots(figsize=(SINGLE, 2.4))
bins = np.linspace(0, 1, 11)
for k, n in enumerate(sel):
    conf = PC[n].max(1); ok = PC[n].argmax(1) == yct
    b = np.clip(np.digitize(conf, bins) - 1, 0, 9)
    xs = [conf[b == j].mean() for j in range(10) if (b == j).sum() >= 5]
    ys = [ok[b == j].mean() for j in range(10) if (b == j).sum() >= 5]
    ece = sum((b == j).mean() * abs(conf[b == j].mean() - ok[b == j].mean()) for j in range(10) if (b == j).any())
    ax.plot(xs, ys, marker="o", ms=3, label=f"{n} (ECE {ece:.3f})", color=["#999999", "#56B4E9", "#0072B2"][k % 3])
ax.plot([0, 1], [0, 1], color="black", ls=":", lw=0.6)
ax.set_xlabel("Confidence"); ax.set_ylabel("Accuracy"); ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.legend(fontsize=5.5, loc="upper left")
save(fig, "fig31_calibration", "Reliability diagram of category predictions on the test set (bins with at least 5 images); ECE = expected calibration error.")

## Index of saved figures

In [ ]:
print(open(os.path.join(FIG_DIR, "captions.md")).read())
print(sorted(f for f in os.listdir(FIG_DIR) if f.endswith(".pdf")))